# Seguridad de la API REST

Endurecemos la API REST del proyecto (`tp_mlops2.api.main`) aplicando el marco **SAIF**
(Secure AI Framework) sobre el servicio real, no sobre un modelo de juguete.

**Riesgo:** la API responde predicciones de demanda elÃ©ctrica a cualquiera que la llame,
sin lÃ­mite. Eso habilita abuso, inyecciÃ³n de payloads malformados, denegaciÃ³n de servicio,
y *scraping* del modelo (reconstruir su comportamiento a fuerza de consultas).

**Controles que agregamos:** autenticaciÃ³n (API key + JWT con rol), *rate limiting*,
validaciÃ³n de entrada (ya existe vÃ­a Pydantic desde la SesiÃ³n 1, la reforzamos), y
un registro de auditorÃ­a como medida de gobernanza.

In [1]:
import os
import time
import datetime

import jwt  # pyjwt
import pandas as pd
from fastapi import Depends, FastAPI, HTTPException
from fastapi.security import APIKeyHeader, HTTPAuthorizationCredentials, HTTPBearer
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

from tp_mlops2.predict import load_model, predict

model, feature_cols, metrics = load_model()
print(f"Modelo cargado: random_forest_demanda. MAE test: {metrics['mae_mw']:.1f} MW")
print(f"Features ({len(feature_cols)}): {feature_cols}")

D:\ESTUDIOS\posgrado\Materias\CEIA (5) Operaciones de aprendizaje de maquina 2\tp_mlops2\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


2026/10/10 18:46:17 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at D:\ESTUDIOS\posgrado\Materias\CEIA (5) Operaciones de aprendizaje de maquina 2\tp_mlops2\.venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Modelo cargado: random_forest_demanda. MAE test: 1736.8 MW
Features (14): ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend', 'temp_Madrid', 'temp_Barcelona', 'temp_Valencia', 'temp_Seville', 'temp_Bilbao', 'load_lag_24h', 'load_lag_168h']


## 1. Estado actual: sin ningÃºn control

AsÃ­ estÃ¡ servida hoy la API real: `/v1/predict` no pide credenciales ni limita cuÃ¡ntas
veces se puede llamar.

In [2]:
app_v = FastAPI()


@app_v.post("/v1/predict")
def predict_v(payload: dict):
    df = pd.DataFrame([payload["features"]], columns=feature_cols)
    pred = predict(model, df, feature_cols)
    return {"predicted_load_mw": float(pred.iloc[0])}


cli_v = TestClient(app_v)
muestra = {
    "features": [0.5, 0.86, 0.0, 1.0, 0.5, 0.86, 0, 28.0, 25.0, 27.0, 32.0, 20.0, 30000, 29500]
}

print("sin credenciales:", cli_v.post("/v1/predict", json=muestra).status_code, "(nadie lo frena)")
codigos = [cli_v.post("/v1/predict", json=muestra).status_code for _ in range(20)]
print(f"20 requests seguidas, todas: {set(codigos)} (sin rate limiting, scraping trivial)")

sin credenciales:

 200 (nadie lo frena)


20 requests seguidas, todas: {200} (sin rate limiting, scraping trivial)


## 2. API key, validaciÃ³n y rate limiting

In [3]:
API_KEY = os.environ.get("MODELO_API_KEY", "demo-key-123")  # en producciÃ³n: solo desde entorno

app = FastAPI(title="API REST demanda elÃ©ctrica (endurecida)")

AUDITORIA: list[dict] = []


def auditar(usuario: str, accion: str, detalle: str) -> None:
    AUDITORIA.append(
        {
            "ts": datetime.datetime.now(datetime.UTC).isoformat(),
            "usuario": usuario,
            "accion": accion,
            "detalle": detalle,
        }
    )


api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)


def auth(k: str = Depends(api_key_header)) -> str:
    if k != API_KEY:
        auditar("desconocido", "auth_fallida", "API key invÃ¡lida o ausente")
        raise HTTPException(401, "no autorizado")
    return "cliente"


_hits: dict[str, list[float]] = {}
LIMITE_POR_MINUTO = 5


def rate_limit(cliente: str = Depends(auth)) -> str:
    now = time.time()
    ventana = [t for t in _hits.get(cliente, []) if now - t < 60]
    if len(ventana) >= LIMITE_POR_MINUTO:
        auditar(cliente, "rate_limit_excedido", f"{len(ventana)} requests en 60s")
        raise HTTPException(429, "demasiadas requests, esperÃ¡ un minuto")
    ventana.append(now)
    _hits[cliente] = ventana
    return cliente


class PredictRequest(BaseModel):
    features: list[float] = Field(min_length=len(feature_cols), max_length=len(feature_cols))


@app.post("/v1/predict")
def predict_endpoint(payload: PredictRequest, cliente: str = Depends(rate_limit)):
    df = pd.DataFrame([payload.features], columns=feature_cols)
    pred = predict(model, df, feature_cols)
    auditar(cliente, "prediccion", f"{pred.iloc[0]:.1f} MW")
    return {"predicted_load_mw": float(pred.iloc[0])}


cli = TestClient(app)
h = {"X-API-Key": API_KEY}

r1 = cli.post("/v1/predict", json=muestra)
r2 = cli.post("/v1/predict", json=muestra, headers=h)
r3 = cli.post("/v1/predict", json={"features": [0.1, 0.2]}, headers=h)  # longitud incorrecta
codigos = [cli.post("/v1/predict", json=muestra, headers=h).status_code for _ in range(7)]

print("sin credenciales :", r1.status_code, "(esperado 401)")
print("con API key      :", r2.status_code, r2.json())
print("payload invÃ¡lido :", r3.status_code, "(esperado 422)")
print("6ta request seguida:", codigos[-1], "(esperado 429)")

sin credenciales : 401 (esperado 401)
con API key      : 200 {'predicted_load_mw': 29556.3055254736}
payload invÃ¡lido : 422 (esperado 422)
6ta request seguida: 429 (esperado 429)


**QuÃ© frena cada control, en tÃ©rminos SAIF:**
- **401 sin credenciales** â†’ acceso no autorizado al modelo de demanda.
- **422 payload invÃ¡lido** â†’ entrada malformada / posible inyecciÃ³n.
- **429 rate limiting** â†’ DoS y *scraping* para reconstruir el modelo por consultas masivas.

## 3. AutorizaciÃ³n por rol con JWT, sobre un endpoint sensible real

In [4]:
JWT_SECRET = os.environ.get("MODELO_JWT_SECRET", "cambiar-en-produccion-con-32-bytes-o-mas")

bearer_scheme = HTTPBearer(auto_error=False)


def crear_token(usuario: str, rol: str, minutos: int = 15) -> str:
    payload = {
        "sub": usuario,
        "rol": rol,
        "exp": datetime.datetime.now(datetime.UTC) + datetime.timedelta(minutes=minutos),
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")


def requiere_rol(*roles_permitidos: str):
    def dependencia(cred: HTTPAuthorizationCredentials = Depends(bearer_scheme)) -> dict:
        if cred is None:
            auditar("desconocido", "auth_fallida", "sin token Bearer")
            raise HTTPException(401, "falta token Bearer")
        try:
            payload = jwt.decode(cred.credentials, JWT_SECRET, algorithms=["HS256"])
        except jwt.PyJWTError as e:
            auditar("desconocido", "auth_fallida", f"token invÃ¡lido: {e}")
            raise HTTPException(401, "token invÃ¡lido o vencido")
        if payload.get("rol") not in roles_permitidos:
            auditar(
                payload.get("sub", "?"),
                "autorizacion_fallida",
                f"rol {payload.get('rol')} insuficiente",
            )
            raise HTTPException(403, "rol insuficiente")
        return payload

    return dependencia


@app.get("/v1/model/metrics")
def model_metrics(usuario: dict = Depends(requiere_rol("admin", "analista"))):
    # endpoint sensible: mÃ©tricas internas del modelo, solo para roles autorizados
    auditar(usuario["sub"], "consulta_metricas", "acceso a /v1/model/metrics")
    return {"mae_mw": metrics["mae_mw"], "mape": metrics["mape"]}


token_analista = crear_token("alan", "analista")
token_invitado = crear_token("visitante", "invitado")

r_ok = cli.get("/v1/model/metrics", headers={"Authorization": f"Bearer {token_analista}"})
r_rol_insuficiente = cli.get("/v1/model/metrics", headers={"Authorization": f"Bearer {token_invitado}"})
r_sin_token = cli.get("/v1/model/metrics")

print("rol analista :", r_ok.status_code, r_ok.json())
print("rol invitado :", r_rol_insuficiente.status_code, "(esperado 403)")
print("sin token    :", r_sin_token.status_code, "(esperado 401)")

print(f"\nEventos de auditorÃ­a registrados: {len(AUDITORIA)}")
for e in AUDITORIA[-5:]:
    print(" ", e["ts"][11:19], e["usuario"], "-", e["accion"], "-", e["detalle"])

rol analista : 200 {'mae_mw': 1736.815230933058, 'mape': 0.05999729717375087}
rol invitado : 403 (esperado 403)
sin token    : 401 (esperado 401)

Eventos de auditorÃ­a registrados: 12
  21:46:22 cliente - rate_limit_excedido - 5 requests en 60s
  21:46:22 cliente - rate_limit_excedido - 5 requests en 60s
  21:46:22 alan - consulta_metricas - acceso a /v1/model/metrics
  21:46:22 visitante - autorizacion_fallida - rol invitado insuficiente
  21:46:22 desconocido - auth_fallida - sin token Bearer


## 4. ReflexiÃ³n

**Â¿QuÃ© amenaza frena el rate limiting y por quÃ© se relaciona con el robo del modelo?**
Frena el *scraping*: consultar el modelo miles de veces con inputs variados para
reconstruir su funciÃ³n de decisiÃ³n sin acceso al artefacto en sÃ­ (extracciÃ³n de modelo).
Limitar requests por minuto hace que ese ataque sea demasiado lento para ser prÃ¡ctico.

**Â¿QuÃ© elemento de SAIF cubre este control?**
AutenticaciÃ³n + autorizaciÃ³n por rol cubre *extender los fundamentos de seguridad al
sistema de IA*; el rate limiting especÃ­fico para un modelo servido por API cubre
*adaptar los controles para mitigar amenazas propias de IA* (no es un control genÃ©rico
de infraestructura, estÃ¡ calibrado al patrÃ³n de abuso de este servicio en particular).

**Â¿Por quÃ© validar la entrada es tambiÃ©n una medida de seguridad y no solo de calidad?**
Un payload con la cantidad incorrecta de features, o valores fuera de rango fÃ­sicamente
imposible, no es solo un error de uso: es la forma mÃ¡s simple de un ataque de inyecciÃ³n
contra el pipeline. Sin validaciÃ³n, esos valores llegarÃ­an directo al modelo, y en un
sistema que reentrena con datos de producciÃ³n (como el DAG de Airflow de este proyecto),
entradas sin validar que se filtren a los datos de reentrenamiento pueden envenenar la
prÃ³xima versiÃ³n del modelo.